# APS Data Cleaning and Preparation



In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

RAW_DIR, PROCESSED_DIR, MODELS_DIR


(WindowsPath('c:/Users/user/Downloads/BSc_Theisis/data/raw'),
 WindowsPath('c:/Users/user/Downloads/BSc_Theisis/data/processed'),
 WindowsPath('c:/Users/user/Downloads/BSc_Theisis/models'))

In [2]:
def load_aps_csv(path: Path) -> pd.DataFrame:
    header_row = None
    with path.open("r", encoding="utf-8") as file_obj:
        for idx, line in enumerate(file_obj):
            if line.startswith("class,"):
                header_row = idx
                break

    if header_row is None:
        raise ValueError(f"Could not find CSV header in {path}")

    return pd.read_csv(path, skiprows=header_row, na_values=["na"])


train_raw = load_aps_csv(RAW_DIR / "aps_failure_training_set.csv")
test_raw = load_aps_csv(RAW_DIR / "aps_failure_test_set.csv")

print("Train shape:", train_raw.shape)
print("Test shape:", test_raw.shape)
train_raw.head()


Train shape: (60000, 171)
Test shape: (16000, 171)


,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


In [3]:
print("Train class distribution:")
print(train_raw["class"].value_counts())

print("\nTest class distribution:")
print(test_raw["class"].value_counts())

train_missing_pct = (train_raw.isna().mean() * 100).sort_values(ascending=False)
print("\nTop 10 missing columns in train (%):")
print(train_missing_pct.head(10).round(2))


Train class distribution:
class
neg    59000
pos     1000
Name: count, dtype: int64

Test class distribution:
class
neg    15625
pos      375
Name: count, dtype: int64

Top 10 missing columns in train (%):
br_000    82.11
bq_000    81.20
bp_000    79.57
bo_000    77.22
ab_000    77.22
cr_000    77.22
bn_000    73.35
bm_000    65.92
bl_000    45.46
bk_000    38.39
dtype: float64


In [4]:
y_train = train_raw["class"].map({"neg": 0, "pos": 1}).astype(int)
y_test = test_raw["class"].map({"neg": 0, "pos": 1}).astype(int)

X_train = train_raw.drop(columns=["class"]).apply(pd.to_numeric, errors="coerce")
X_test = test_raw.drop(columns=["class"]).apply(pd.to_numeric, errors="coerce")

missing_threshold = 0.70
missing_fraction = X_train.isna().mean()
columns_to_drop = missing_fraction[missing_fraction > missing_threshold].index.tolist()

X_train = X_train.drop(columns=columns_to_drop)
X_test = X_test.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} columns with > {missing_threshold:.0%} missing values")
print(columns_to_drop)
print("Feature count after dropping columns:", X_train.shape[1])


Dropped 7 columns with > 70% missing values
['ab_000', 'bn_000', 'bo_000', 'bp_000', 'bq_000', 'br_000', 'cr_000']
Feature count after dropping columns: 163


In [5]:
imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imputed),
    columns=X_train_imputed.columns,
    index=X_train_imputed.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imputed),
    columns=X_test_imputed.columns,
    index=X_test_imputed.index,
)

train_clean = X_train_scaled.copy()
train_clean["class"] = y_train.values

test_clean = X_test_scaled.copy()
test_clean["class"] = y_test.values

print("Train clean shape:", train_clean.shape)
print("Test clean shape:", test_clean.shape)
print("Any missing values left in train:", train_clean.isna().any().any())
print("Any missing values left in test:", test_clean.isna().any().any())


Train clean shape: (60000, 164)
Test clean shape: (16000, 164)
Any missing values left in train: False
Any missing values left in test: False


In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_output = PROCESSED_DIR / "aps_train_cleaned.csv"
test_output = PROCESSED_DIR / "aps_test_cleaned.csv"
artifact_output = MODELS_DIR / "aps_preprocessing.joblib"

train_clean.to_csv(train_output, index=False)
test_clean.to_csv(test_output, index=False)

joblib.dump(
    {
        "columns_to_drop": columns_to_drop,
        "feature_names": X_train_imputed.columns.tolist(),
        "imputer": imputer,
        "scaler": scaler,
        "missing_threshold": missing_threshold,
    },
    artifact_output,
)

print("Saved:")
print(train_output)
print(test_output)
print(artifact_output)


Saved:
c:\Users\user\Downloads\BSc_Theisis\data\processed\aps_train_cleaned.csv
c:\Users\user\Downloads\BSc_Theisis\data\processed\aps_test_cleaned.csv
c:\Users\user\Downloads\BSc_Theisis\models\aps_preprocessing.joblib
